# What actually moved the score on S6E8 — and what didn't

### Playground S6E8 · ROC AUC · one big win, nine dead ends

**Everything below trains from `train.csv` alone.** That costs about 0.0013 against the top
of the public leaderboard — which is mostly stacked from other competitors' prediction
files — and buys a number you can fork and reproduce.

I tested 30+ ideas on this competition. **Two were worth anything. Most of the rest were
worth nothing.** This notebook shows both, because knowing which well-motivated ideas fail
on a dataset is the part that transfers.

Final: **public LB 0.96990**.

**The two that worked:** target-encoding every feature — including the continuous ones —
as a high-cardinality categorical (§6), and the **decimal lattice** the generator left in
the numbers (§7). Between them they were worth more than feature engineering, tuning,
ensembling and model selection put together.

**Built with AI.** I worked through this with [Claude Code](https://claude.com/claude-code)
(Opus). I chose the competition, supplied a playbook from my previous Playground runs,
set up the environment, and decided what to submit; the agent did the EDA, wrote the
experiments, and analysed results. Worth saying plainly: the winning idea came from
[a public notebook by OMID BAGHCHEH SARAEI](https://www.kaggle.com/code/omidbaghchehsaraei/tabm-for-predicting-smartphone-addiction),
not from our own search. The agent's contribution was running every experiment properly —
including the ones that failed — rather than only the interesting ones.

Runtime ~35 min on a T4. Every number below is computed live.

## 1. Setup

In [1]:
import os, glob, time, warnings
import numpy as np, pandas as pd
import xgboost as xgb, lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, r2_score
warnings.filterwarnings("ignore")

def find_first(*pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h: return h[0]

train_csv = find_first("/kaggle/input/competitions/playground-series-s6e8/train.csv",
                       "/kaggle/input/playground-series-s6e8/train.csv",
                       "/kaggle/input/*/train.csv", "/kaggle/input/**/train.csv",
                       "playground-series-s6e8/train.csv",
                       "../playground-series-s6e8/train.csv")
assert train_csv, "attach the playground-series-s6e8 competition via '+ Add Input'"
DATA = os.path.dirname(train_csv)
ORIG_CSV = find_first("/kaggle/input/**/Smartphone_Usage_And_Addiction*.csv",
                      "../original/**/Smartphone_Usage_And_Addiction*.csv")

DEVICE = "cuda" if os.path.exists("/proc/driver/nvidia") or os.environ.get("CUDA_PATH") else "cpu"
try:
    xgb.XGBClassifier(device="cuda", tree_method="hist", n_estimators=1).fit(
        np.zeros((8, 2)), [0, 1] * 4); DEVICE = "cuda"
except Exception:
    DEVICE = "cpu"

TARGET = "addicted_label"
CATS = ["gender", "stress_level", "academic_work_impact"]
NUMS = ["age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
        "work_study_hours", "sleep_hours", "notifications_per_day",
        "app_opens_per_day", "weekend_screen_time"]

train = pd.read_csv(f"{DATA}/train.csv"); test = pd.read_csv(f"{DATA}/test.csv")
y = train[TARGET].values
print(f"train {train.shape}  test {test.shape}  positive rate {y.mean():.4f}  device {DEVICE}")

train (691369, 14)  test (296302, 13)  positive rate 0.7094  device cuda


## 2. Measure your noise floor before believing anything

The spread between folds inside one 5-fold run is **not** your uncertainty. It badly
overstates how much the *mean* moves under a different partition.

Repeat the whole 5-fold CV with different partition seeds and take the std of the
repeated **means**. That's the bar every idea has to clear.

In [2]:
XGB_BASE = dict(n_estimators=4000, learning_rate=0.05, max_depth=6, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=20, tree_method="hist",
                device=DEVICE, enable_categorical=True, eval_metric="auc",
                early_stopping_rounds=100)
mk = lambda seed, **o: xgb.XGBClassifier(**{**XGB_BASE, "random_state": seed, **o})

def cv_once(X, y, seed=42, model_fn=None):
    oof = np.zeros(len(X)); folds = []
    for itr, iva in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y):
        m = (model_fn or mk)(seed)
        m.fit(X.iloc[itr], y[itr], eval_set=[(X.iloc[iva], y[iva])], verbose=False)
        oof[iva] = m.predict_proba(X.iloc[iva])[:, 1]
        folds.append(roc_auc_score(y[iva], oof[iva]))
    return roc_auc_score(y, oof), oof, folds

def repeated_cv(X, y, label, seeds=(42, 2024, 7), model_fn=None):
    t0 = time.time(); s = [cv_once(X, y, sd, model_fn)[0] for sd in seeds]
    s = np.array(s)
    print(f"  {label:<32s} {s.mean():.5f} +/- {s.std(ddof=1):.5f}  ({time.time()-t0:.0f}s)")
    return s

def fe_raw(df):
    X = df.drop(columns=[TARGET, "id"], errors="ignore").copy()
    for c in CATS: X[c] = X[c].astype("category")
    return X

Xr = fe_raw(train)
_, _, folds = cv_once(Xr, y, 42)
print("per-fold AUCs in ONE run:", " ".join(f"{f:.5f}" for f in folds))
print(f"per-fold range = {max(folds)-min(folds):.5f}   <-- NOT the noise floor\n")
raw_scores = repeated_cv(Xr, y, "raw features, 3 partition seeds")
FLOOR = raw_scores.std(ddof=1)
print(f"\nNOISE FLOOR = {FLOOR:.5f}  ({(max(folds)-min(folds))/FLOOR:.0f}x tighter "
      f"than the per-fold range)")

per-fold AUCs in ONE run: 0.96368 0.96435 0.96459 0.96519 0.96415
per-fold range = 0.00151   <-- NOT the noise floor

  raw features, 3 partition seeds  0.96437 +/- 0.00004  (182s)

NOISE FLOOR = 0.00004  (40x tighter than the per-fold range)


## 3. The structure in the data — and a story I got wrong

Screen time decomposes exactly: `daily = social + gaming + work + other`, with `other >= 0`
and **zero violations**. That's a generator invariant, not a correlation.

In [3]:
ACC = ["daily_screen_time_hours", "social_media_hours", "gaming_hours", "work_study_hours"]
cp = train.dropna(subset=ACC)
resid = cp.daily_screen_time_hours - cp[ACC[1:]].sum(axis=1)
print(f"complete rows {len(cp):,}   min residual {resid.min():.6f}   "
      f"violations {(resid < -1e-9).sum():,}")

m = train.work_study_hours.notna()
print(f"\nwork_study_hours marginal AUC = "
      f"{roc_auc_score(y[m.values], train.loc[m,'work_study_hours']):.4f}  (looks harmful)")
q = pd.qcut(train.daily_screen_time_hours, 5, labels=False, duplicates="drop")
qw = pd.qcut(train.work_study_hours, 5, labels=False, duplicates="drop")
grid = train.groupby([q, qw], observed=True)[TARGET].mean().unstack()
print("\naddiction rate, rows = daily_screen quintile, cols = work_study quintile:")
print(grid.round(3).to_string())

complete rows 421,427   min residual 0.000000   violations 0

work_study_hours marginal AUC = 0.6549  (looks harmful)

addiction rate, rows = daily_screen quintile, cols = work_study quintile:
work_study_hours           0.0    1.0    2.0    3.0    4.0
daily_screen_time_hours                                   
0.0                      0.358  0.285  0.125  0.031  0.006
1.0                      0.568  0.550  0.482  0.307  0.198
2.0                      0.842  0.836  0.843  0.816  0.704
3.0                      0.983  0.987  0.985  0.988  0.981
4.0                      1.000  0.999  0.999  0.999  0.999


Marginally `work_study_hours` looks like a risk factor. Hold total screen time fixed and
it flips: in the lowest screen-time quintile the rate falls from ~0.36 to ~0.005 as
work/study rises. A classic Simpson's reversal, invisible to a correlation matrix.

**My first explanation was wrong.** I said work/study screen time is "productive" screen
time, so conditioning reveals a protective effect. Then I checked the real-world dataset
the competition was generated from (§4): there, `work_study_hours` has AUC **0.5007** —
pure noise. There is no protective effect to reveal. The reversal is pure arithmetic: the
generator builds `daily` as a sum, so conditioning on the total makes the parts informative
about how it splits.

The feature engineering still worked. The explanation was invented after the fact.

## 4. Check the source dataset — as a diagnostic, not for extra rows

Playground data is generated from a real dataset. Concatenating it is standard practice.
Here it was worth **−0.00008** as training data — and worth a lot as evidence.

In [4]:
if ORIG_CSV is None:
    print("Attach 'jayjoshi37/smartphone-usage-and-addiction-prediction' to run this.")
else:
    orig = pd.read_csv(ORIG_CSV)
    r_o = orig.daily_screen_time_hours - orig[ACC[1:]].sum(axis=1)
    print(f"the accounting identity is violated in:")
    print(f"   synthetic (this competition): {(resid < -1e-9).mean():6.1%} of rows")
    print(f"   original (real data)        : {(r_o   < -1e-9).mean():6.1%} of rows\n")
    cmp = pd.DataFrame([dict(feature=c,
                             real=roc_auc_score(orig.addicted_label, orig[c]),
                             synthetic=roc_auc_score(y[train[c].notna().values],
                                                     train.loc[train[c].notna(), c]))
                        for c in NUMS])
    cmp["gap"] = cmp.synthetic - cmp.real
    print(cmp.sort_values("gap", ascending=False).head(5)
             .to_string(index=False, float_format="%.4f"))

the accounting identity is violated in:
   synthetic (this competition):   0.0% of rows
   original (real data)        :  60.7% of rows

            feature   real  synthetic    gap
   work_study_hours 0.5007     0.6549 0.1541
       gaming_hours 0.5054     0.6220 0.1166
 social_media_hours 0.7634     0.8578 0.0944
  app_opens_per_day 0.5070     0.5409 0.0339
weekend_screen_time 0.8558     0.8810 0.0252


The identity does not exist in the real data — it holds in 100% of competition rows and is
violated in **60.7%** of the source. And `work_study_hours` (0.5007) and `gaming_hours`
(0.5054) are **pure noise in reality** but predictive here.

The generator manufactured the structure. Exploit it — we're scored on the synthetic
distribution — but don't explain it in real-world terms.

## 5. Imputation: augment, don't replace

Every column is 4–20% missing. Ratio features are NaN whenever any input is missing (39%
of rows), so imputing should unlock them. We tried it two ways.

In [5]:
IMP = dict(n_estimators=400, learning_rate=0.08, max_depth=6, subsample=0.8,
           colsample_bytree=0.8, min_child_weight=20, tree_method="hist",
           device=DEVICE, enable_categorical=True)

def impute(tr_, te_, seed=42):
    """One XGB regressor per column, fit on train+test together. No target involved,
    so this is transductive preprocessing, not leakage."""
    n = len(tr_); full = pd.concat([tr_[NUMS+CATS], te_[NUMS+CATS]], ignore_index=True)
    X = full.copy()
    for c in CATS: X[c] = X[c].astype("category")
    out = full[NUMS].copy()
    for col in NUMS:
        obs = X[col].notna().values
        feats = [c for c in NUMS+CATS if c != col]
        m = xgb.XGBRegressor(**IMP, random_state=seed).fit(X.loc[obs, feats], X.loc[obs, col])
        if (~obs).sum(): out.loc[~obs, col] = m.predict(X.loc[~obs, feats])
    return out.iloc[:n].reset_index(drop=True), out.iloc[n:].reset_index(drop=True)

t0 = time.time(); tr_imp, te_imp = impute(train, test)
print(f"imputers fitted ({time.time()-t0:.0f}s)")

def fe(imp, orig, keep_raw):
    """Composition features on imputed values + missingness flags.
    keep_raw=True also keeps the ORIGINAL NaN-bearing columns alongside."""
    X = imp.copy()
    d, s, g = X.daily_screen_time_hours, X.social_media_hours, X.gaming_hours
    w, wk, sl = X.work_study_hours, X.weekend_screen_time, X.sleep_hours
    n, o = X.notifications_per_day, X.app_opens_per_day
    parts = s + g + w
    X["resid"], X["leisure"] = d - parts, d - w
    X["social_frac"], X["work_frac"] = s/d, w/d
    X["leisure_frac"], X["resid_frac"] = (d-w)/d, (d-parts)/d
    X["wk_ratio"], X["week_total"] = wk/d, 5*d + 2*wk
    X["awake_screen_frac"], X["free_time"] = d/(24-sl), 24 - sl - d - w
    X["notif_per_open"], X["min_per_open"] = n/o, d*60/o
    for c in CATS: X[c] = orig[c].astype("category").values
    for c in NUMS + CATS: X[f"na_{c}"] = orig[c].isna().astype(np.int8).values
    if keep_raw:
        for c in NUMS: X[f"rawnan_{c}"] = orig[c].values
    return X

X_replace = fe(tr_imp, train, False)
X_augment = fe(tr_imp, train, True)
print(f"replace: {X_replace.shape[1]} features   augment: {X_augment.shape[1]} features\n")
rep = repeated_cv(X_replace, y, "imputed REPLACES the NaNs")
aug = repeated_cv(X_augment, y, "imputed ALONGSIDE the NaNs")
print(f"\nreplace: {rep.mean()-raw_scores.mean():+.5f} vs raw   "
      f"({(rep.mean()-raw_scores.mean())/FLOOR:+.0f}x floor)")
print(f"augment: {aug.mean()-raw_scores.mean():+.5f} vs raw   "
      f"({(aug.mean()-raw_scores.mean())/FLOOR:+.0f}x floor)")

imputers fitted (25s)
replace: 36 features   augment: 45 features

  imputed REPLACES the NaNs        0.96348 +/- 0.00001  (228s)
  imputed ALONGSIDE the NaNs       0.96562 +/- 0.00002  (280s)

replace: -0.00090 vs raw   (-23x floor)
augment: +0.00125 vs raw   (+33x floor)


**Same imputer, opposite sign.**

Replacing hurts because a GBM's native NaN handling learns a *default split direction* per
node — strictly more expressive than one imputed point estimate, which drags the row toward
the middle of the distribution. Keeping both columns gives the model the missingness
structure *and* full ratio coverage.

> For a NaN-native model, an imputed column should be an **extra feature, never a
> substitute**. The fact that a value is missing is information.

## 6. The one that mattered: target-encode everything, continuous columns included

Cast every column to a string level, then replace it with (a) the smoothed mean of the
target for that level and (b) the level's frequency.

That sounds reckless for a continuous column. It works here because of the numbers:
`daily_screen_time_hours` has 1,389 distinct values over 691k rows — **~500 rows per
level** — so a smoothed target mean is a well-estimated quantity at 1,389 points. A tree
approximates the same steep curve with a few dozen splits.

**Check this before copying the idea:** it needs high `n` per level. Run the cell below on
your own data first.

**Leakage is the entire difficulty.** The encoding uses the target, so a careless version
inflates CV and collapses on the leaderboard. This uses a nested scheme: for each outer
fold, statistics come only from that fold's training portion, and rows *within* that
portion get inner out-of-fold encodings. No row's encoding ever sees its own target.

In [6]:
ENC_COLS = NUMS + CATS
print(f"{'column':<26s}{'levels':>8s}{'rows/level':>12s}")
for c in ENC_COLS:
    k = train[c].astype(str).nunique()
    print(f"{c:<26s}{k:>8,d}{len(train)/k:>12,.0f}")

SMOOTH = 10.0
LTR = pd.DataFrame({c: train[c].astype(str).values for c in ENC_COLS})
LTE = pd.DataFrame({c: test[c].astype(str).values for c in ENC_COLS})
ORDER = [f"te_{c}" for c in ENC_COLS] + [f"fq_{c}" for c in ENC_COLS]

def maps_from(levels, yy):
    gm = yy.mean(); m = {}
    for c in ENC_COLS:
        g = pd.DataFrame({"lv": levels[c].values, "y": yy}).groupby("lv")["y"].agg(["count","mean"])
        m[c] = (((g["count"]*g["mean"] + SMOOTH*gm)/(g["count"]+SMOOTH)).astype(np.float32),
                g["count"].astype(np.float32))
    return m, gm

def apply_maps(levels, m, gm):
    out = {}
    for c in ENC_COLS:
        tmap, fmap = m[c]
        out[f"te_{c}"] = levels[c].map(tmap).astype(np.float32).fillna(gm).values
        out[f"fq_{c}"] = levels[c].map(fmap).astype(np.float32).fillna(0.0).values
    return pd.DataFrame(out)[ORDER]

X_te_base = X_augment.reset_index(drop=True)
X_te_base_test = fe(te_imp, test, True).reset_index(drop=True)
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X_te_base, y))

def build_enc(itr, iva):
    y_tr = y[itr]; L = LTR.iloc[itr].reset_index(drop=True)
    holder = np.zeros((len(itr), len(ORDER)), dtype=np.float32)
    for i_in, i_out in StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(len(itr)), y_tr):
        m, gm = maps_from(L.iloc[i_in], y_tr[i_in])
        holder[i_out] = apply_maps(L.iloc[i_out].reset_index(drop=True), m, gm).values
    m, gm = maps_from(L, y_tr)
    return (pd.DataFrame(holder, columns=ORDER),
            apply_maps(LTR.iloc[iva].reset_index(drop=True), m, gm),
            apply_maps(LTE, m, gm))

column                      levels  rows/level
age                             19      36,388
daily_screen_time_hours      1,390         497
social_media_hours             722         958
gaming_hours                   402       1,720
work_study_hours               601       1,150
sleep_hours                    452       1,530
notifications_per_day          232       2,980
app_opens_per_day              167       4,140
weekend_screen_time          1,438         481
gender                           4     172,842
stress_level                     4     172,842
academic_work_impact             3     230,456


In [7]:
def run_te(use_enc, label, save=None, extra_tr=None, extra_te=None):
    t0 = time.time(); oof = np.zeros(len(X_te_base)); tp = np.zeros(len(X_te_base_test))
    for f, (itr, iva) in enumerate(FOLDS):
        Xa = X_te_base.iloc[itr].reset_index(drop=True)
        Xb = X_te_base.iloc[iva].reset_index(drop=True)
        Xt = X_te_base_test
        if use_enc:
            e_tr, e_va, e_te = build_enc(itr, iva)
            Xa = pd.concat([Xa, e_tr], axis=1); Xb = pd.concat([Xb, e_va], axis=1)
            Xt = pd.concat([Xt, e_te], axis=1)
        if extra_tr is not None:
            Xa = pd.concat([Xa, extra_tr.iloc[itr].reset_index(drop=True)], axis=1)
            Xb = pd.concat([Xb, extra_tr.iloc[iva].reset_index(drop=True)], axis=1)
            Xt = pd.concat([Xt, extra_te], axis=1)
        m = mk(42); m.fit(Xa, y[itr], eval_set=[(Xb, y[iva])], verbose=False)
        oof[iva] = m.predict_proba(Xb)[:, 1]; tp += m.predict_proba(Xt)[:, 1] / 5
    a = roc_auc_score(y, oof)
    print(f"  {label:<34s} OOF {a:.5f}  ({time.time()-t0:.0f}s)")
    if save: PRED[save] = (oof, tp)
    return a

PRED = {}
a_no = run_te(False, "no encodings (imputed+composition)", save="xgb_aug")
a_te = run_te(True,  "+ target & frequency encodings",     save="xgb_te")
print(f"\ndelta = {a_te - a_no:+.5f}   ({(a_te-a_no)/FLOOR:+.0f}x noise floor)")

  no encodings (imputed+composition) OOF 0.96560  (101s)
  + target & frequency encodings     OOF 0.96801  (106s)

delta = +0.00241   (+63x noise floor)


That single change is larger than everything else in this notebook combined.

**How we checked it wasn't leaking:** predict the leaderboard before submitting. Our
measured CV→LB offset is about +0.00125, so a clean encoding should land ~0.9697. It
scored **0.96942**. A leaky encoding shows up as CV/LB *divergence*, not a uniformly bad
score — so the agreement is the evidence.

**Nine variations on this idea all failed** (measured, same folds):

| variation | delta |
|---|---|
| pairwise TE, 36 pairs x 32 bins | −0.00040 |
| pairwise TE, 6 pairs x 10 bins, heavy smoothing | −0.00007 |
| multi-resolution TE (raw + 300/100/30 bins) | −0.00033 |
| smoothing 50 / 200 instead of 10 | −0.00006 / −0.00030 |
| adaptive per-column smoothing | +0.00002 |

Plain single-feature TE at smoothing 10 is the sweet spot. Once the representation was
right, nothing else moved.

## 7. The decimal lattice: what the digits know that the values don't

At this point I thought the dataset was finished. Target encoding had landed, and a dozen
follow-ups — pairwise encodings, multi-resolution keys, smoothing sweeps, k-NN target
features, extra model families — all returned nothing.

There is even a way to *check* that formally. This target is a Bernoulli draw from a smooth
probability field (out-of-fold predictions land on the calibration diagonal), which means
there is a hard AUC ceiling no model can pass, and a calibrated model lets you estimate it:

$$\text{AUC}^{*} = \frac{\mathbb{E}\left[\mathbb{1}\{p_i > p_j\}\, p_i (1-p_j)\right]}
{\mathbb{E}[p]\,\mathbb{E}[1-p]}$$

Computed on our stack it said **+0.000024 of room left** — half the noise floor. Which
looked like proof we were done.

It wasn't. Look at the first decimal digit of `daily_screen_time_hours`:

In [8]:
d1 = np.floor(train["daily_screen_time_hours"].values * 10) % 10
m = ~np.isnan(d1)
tbl = pd.DataFrame({"first decimal": d1[m], "y": y[m]}).groupby("first decimal")["y"] \
        .agg(rate="mean", rows="size")
print(tbl.to_string(float_format="%.4f"))
print(f"\nspread across digits: {tbl['rate'].max() - tbl['rate'].min():.4f}   "
      f"(base rate {y.mean():.4f})")

                rate   rows
first decimal              
0.0           0.6513  61155
1.0           0.7044  55137
2.0           0.7365  65419
3.0           0.7188  68386
4.0           0.6721  51975
5.0           0.7161  58964
6.0           0.7117  61884
7.0           0.7192  62862
8.0           0.7239  58673
9.0           0.7326  51060

spread across digits: 0.0852   (base rate 0.7094)


An **8.5-point swing in the addiction rate depending on the first decimal digit**, across
50–68k rows per digit. Nothing about a person's behaviour explains that. It is a
fingerprint of *how the generator produced the numbers*.

And target encoding cannot see it. TE estimates every exact value independently — it has no
way to say "everything ending in .2 shares something," because that statement pools across
integer parts and TE's levels don't. Different channel, not a different view of the same
one.

In [9]:
FRAC_COLS = ["daily_screen_time_hours", "social_media_hours", "gaming_hours",
             "work_study_hours", "sleep_hours", "weekend_screen_time"]

def lattice(df):
    o = {}
    for c in FRAC_COLS:
        v = df[c].values
        o[f"frac_{c}"] = v - np.floor(v)        # sub-unit position, continuous
        o[f"d1_{c}"]   = np.floor(v * 10) % 10  # first decimal digit
    return pd.DataFrame(o).astype(np.float32)

LAT_TR, LAT_TE = lattice(train), lattice(test)
a_lat = run_te(True, "+ decimal lattice (frac, d1)", save="xgb_lattice",
               extra_tr=LAT_TR, extra_te=LAT_TE)
print(f"\ndelta vs target encoding alone = {a_lat - a_te:+.5f}   "
      f"({(a_lat-a_te)/FLOOR:+.1f}x noise floor)")
print(f"for scale, the estimated room to the ceiling was +0.000024")

  + decimal lattice (frac, d1)       OOF 0.96810  (114s)

delta vs target encoding alone = +0.00009   (+2.3x noise floor)
for scale, the estimated room to the ceiling was +0.000024


### What that means, and one trap

The gain is small in absolute terms but it is **several times the headroom the ceiling
calculation said existed.** So the ceiling estimate was wrong — in a specific, useful way.

It measures *no signal reachable from the current representation*. That claim was correct:
tuning and ensembling really were exhausted, and a dozen experiments had already shown it.
What it cannot bound is what a **new channel** finds. Use it to stop tuning; never to stop
looking.

The distinction that predicts which ideas escape it:

| | example | result |
|---|---|---|
| a **new channel** — information absent from the features | the decimal lattice | **+0.00011** |
| a **new view of the same channel** | k-NN target encoding (0.936 standalone!) | +0.00001 |

**The trap.** I then tested the *second* decimal digit, because a conditional control said
it had 1.5x the spread of the first. It added nothing. The control was measured in 25 value
bands — but each band spans ~56 distinct values, so within a band the second digit nearly
*determines* the exact value, which TE already encodes. The control was still measuring the
thing it was supposed to be holding fixed. **A conditional control needs enough resolution
that the conditioned variable stops proxying the control** — otherwise it will hand you a
confident wrong answer, as it did me.

## 8. Stacking: use logits, and let the combiner subtract

Two combiners over a small library:

- **Hill climbing** — greedy forward selection. Builds a weighted average, so every weight
  is >= 0 by construction.
- **Logit stacking** — logistic regression on `clip(log(p/(1-p)), ±30)`, which *can* go
  negative.

Both are fit on the OOF matrix, so scoring them on it reads high. Each is fit on half the
OOF rows and scored on the other half, over 5 splits, with **paired** differences.

In [10]:
def member(name, kind, rep):
    t0 = time.time(); oof = np.zeros(len(X_te_base)); tp = np.zeros(len(X_te_base_test))
    for f, (itr, iva) in enumerate(FOLDS):
        Xa = X_te_base.iloc[itr].reset_index(drop=True)
        Xb = X_te_base.iloc[iva].reset_index(drop=True); Xt = X_te_base_test
        if rep == "te":
            e_tr, e_va, e_te = build_enc(itr, iva)
            Xa = pd.concat([Xa, e_tr], axis=1); Xb = pd.concat([Xb, e_va], axis=1)
            Xt = pd.concat([Xt, e_te], axis=1)
        if kind == "lgb":
            m = lgb.LGBMClassifier(n_estimators=4000, learning_rate=0.05, num_leaves=63,
                                   colsample_bytree=.8, subsample=.8, subsample_freq=1,
                                   min_child_samples=100, verbose=-1, random_state=42)
            m.fit(Xa, y[itr], eval_set=[(Xb, y[iva])], eval_metric="auc",
                  callbacks=[lgb.early_stopping(100, verbose=False)])
        else:
            Xa, Xb, Xt = [d.copy() for d in (Xa, Xb, Xt)]
            cc = [c for c in Xa.columns if isinstance(Xa[c].dtype, pd.CategoricalDtype)]
            for d in (Xa, Xb, Xt):
                for c in cc: d[c] = d[c].astype(object).fillna("__m__").astype(str)
            m = CatBoostClassifier(iterations=3000, learning_rate=0.06, depth=6,
                                   eval_metric="AUC", verbose=0, random_seed=42,
                                   early_stopping_rounds=100,
                                   **({"task_type": "GPU"} if DEVICE == "cuda" else {}))
            m.fit(Xa, y[itr], eval_set=(Xb, y[iva]),
                  cat_features=[Xa.columns.get_loc(c) for c in cc], verbose=0)
        oof[iva] = m.predict_proba(Xb)[:, 1]; tp += m.predict_proba(Xt)[:, 1] / 5
    PRED[name] = (oof, tp)
    print(f"  {name:<12s} {roc_auc_score(y, oof):.5f}  ({time.time()-t0:.0f}s)")

member("lgb_te", "lgb", "te")
member("cat_aug", "cat", "aug")   # weak + different: watch what the stacker does with it
print()
for k, (o, _) in PRED.items(): print(f"  {k:<12s} {roc_auc_score(y, o):.5f}")

  lgb_te       0.96775  (444s)


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  cat_aug      0.96163  (542s)

  xgb_aug      0.96560
  xgb_te       0.96801
  xgb_lattice  0.96810
  lgb_te       0.96775
  cat_aug      0.96163


In [11]:
def to_logit(p, clip=30.0):
    p = np.clip(np.asarray(p, np.float64), 1e-15, 1-1e-15)
    return np.clip(np.log(p/(1-p)), -clip, clip)

def hill_climb(o, yy, n_iter=30):
    nm = list(o); P = np.column_stack([o[n] for n in nm]); picks = np.zeros(len(nm), int)
    j = int(np.argmax([roc_auc_score(yy, P[:, k]) for k in range(P.shape[1])]))
    picks[j] = 1; cur = P[:, j].copy(); best = roc_auc_score(yy, cur); k = 1
    for _ in range(n_iter):
        cb, cj = best, -1
        for t in range(P.shape[1]):
            a = roc_auc_score(yy, (cur*k + P[:, t])/(k+1))
            if a > cb + 1e-9: cb, cj = a, t
        if cj < 0: break
        cur = (cur*k + P[:, cj])/(k+1); k += 1; picks[cj] += 1; best = cb
    s = picks.sum() or 1
    return {n: v/s for n, v in zip(nm, picks)}

names = list(PRED)
Z = np.column_stack([to_logit(PRED[n][0]) for n in names])
R = np.column_stack([PRED[n][0] for n in names])
rows = []
for rep in range(5):
    iA, iB = next(StratifiedShuffleSplit(1, test_size=.5, random_state=rep)
                  .split(np.zeros(len(y)), y))
    w = hill_climb({n: PRED[n][0][iA] for n in names}, y[iA])
    rows.append(dict(
        best_solo=max(roc_auc_score(y[iB], PRED[n][0][iB]) for n in names),
        hill_climb=roc_auc_score(y[iB], sum(w[n]*PRED[n][0][iB] for n in names)),
        logit_stack=roc_auc_score(y[iB], LogisticRegression(max_iter=2000)
                                  .fit(Z[iA], y[iA]).predict_proba(Z[iB])[:, 1]),
        raw_stack=roc_auc_score(y[iB], LogisticRegression(max_iter=2000)
                                .fit(R[iA], y[iA]).predict_proba(R[iB])[:, 1]),
        mean_all=roc_auc_score(y[iB], R[iB].mean(1))))
rob = pd.DataFrame(rows)
print(rob.to_string(float_format="%.6f"))
print("\nPAIRED DIFFERENCES (same rows, so split noise cancels)")
for a, b in [("logit_stack","hill_climb"), ("logit_stack","raw_stack"),
             ("logit_stack","best_solo"), ("mean_all","best_solo")]:
    d = rob[a] - rob[b]
    ok = "consistent" if (d > 0).all() or (d < 0).all() else "SIGN FLIPS"
    print(f"  {a:>12s} - {b:<12s} = {d.mean():+.6f} +/- {d.std(ddof=1):.6f}  [{ok}]")

meta = LogisticRegression(max_iter=2000).fit(Z, y)
w_full = hill_climb({n: PRED[n][0] for n in names}, y)
print("\n" + pd.DataFrame({"solo": [roc_auc_score(y, PRED[n][0]) for n in names],
                            "hill_climb_w": [w_full[n] for n in names],
                            "stacker_coef": meta.coef_[0]}, index=names)
      .sort_values("stacker_coef", ascending=False).to_string(float_format="%.4f"))

   best_solo  hill_climb  logit_stack  raw_stack  mean_all
0   0.968183    0.968569     0.968617   0.968453  0.968204
1   0.968442    0.968831     0.968883   0.968708  0.968465
2   0.968478    0.968830     0.968888   0.968675  0.968429
3   0.967869    0.968243     0.968294   0.968120  0.967865
4   0.967886    0.968288     0.968330   0.968172  0.967927

PAIRED DIFFERENCES (same rows, so split noise cancels)
   logit_stack - hill_climb   = +0.000050 +/- 0.000006  [consistent]
   logit_stack - raw_stack    = +0.000177 +/- 0.000022  [consistent]
   logit_stack - best_solo    = +0.000431 +/- 0.000014  [consistent]
      mean_all - best_solo    = +0.000006 +/- 0.000035  [SIGN FLIPS]

              solo  hill_climb_w  stacker_coef
xgb_lattice 0.9681        0.4000        0.3899
xgb_aug     0.9656        0.2000        0.3475
xgb_te      0.9680        0.2000        0.2365
lgb_te      0.9677        0.2000        0.1527
cat_aug     0.9616        0.0000       -0.1121


Two things to take from that table.

**Hill climbing can only add; a stacker can subtract.** The weak member gets zero
hill-climbing weight and a *negative* stacker coefficient. Weak-but-decorrelated models
carry usable information as **corrections**, not as things to average in. If a hill
climber zeroes out a member, try a linear stacker before discarding it.

**How much worse a plain average is depends on your worst member.** In the small library
above the members are close together, and `mean_all` comes out slightly *ahead* of the best
single model — but still far behind the stacker. In our full 12-model library, which
included two neural nets at 0.944, the same plain average was **0.0012 worse than simply
using the best member**. An unweighted scheme has no way to say a member is bad, so its
cost scales with how bad the worst one is. Check the printed number above rather than
assuming either direction.

**Stack on logits, not probabilities.** How much this matters depends on your library: it
was worth +0.00047 in our full 12-model library and only +0.00008 in an all-GBM one.
Boosted trees agree in the saturated region, so there's less for the logit scale to
recover. This target saturates hard — the top screen-time decile has an addiction rate of
1.000, where probabilities have no resolution left and logits still do.

## 9. Submission

In [12]:
Zt = np.column_stack([to_logit(PRED[n][1]) for n in names])
pred = meta.predict_proba(Zt)[:, 1]
mo = np.zeros(len(y))
for itr, iva in StratifiedKFold(5, shuffle=True, random_state=42).split(Z, y):
    mo[iva] = LogisticRegression(max_iter=2000).fit(Z[itr], y[itr]).predict_proba(Z[iva])[:, 1]
print(f"cross-fitted stack OOF = {roc_auc_score(y, mo):.6f}")
sub = pd.DataFrame({"id": test["id"].values, TARGET: pred})
sub.to_csv("submission.csv", index=False)
print(f"submission.csv  rows={len(sub):,}  mean {pred.mean():.4f}  "
      f"(train rate {y.mean():.4f})")
sub.head()

cross-fitted stack OOF = 0.968512
submission.csv  rows=296,302  mean 0.7093  (train rate 0.7094)


,id,addicted_label
0,691369,0.999845
1,691370,0.957460
2,691371,0.851541
3,691372,0.995336
4,691373,0.998467


**CV does not estimate your leaderboard score — but CV *differences* do.**

Our two full submissions: CV 0.966012 → LB 0.96751, and CV 0.966217 → LB 0.96769. Both
beat their CV by ~0.0013, because every test prediction is an average of 5 fold-models
while every OOF prediction comes from one. But the *difference* between the pipelines was
−0.000205 on CV and −0.00018 on the LB.

Trust CV for ranking decisions, never as a leaderboard estimate. An offset is harmless; a
wrong ordering is not.

## 10. Everything tried

| idea | delta | verdict |
|---|---|---|
| **target + frequency encoding, all columns** | **+0.0023 CV / +0.0017 LB** | ✅ the win |
| imputed columns *alongside* the NaNs | +0.0012 | ✅ |
| **decimal lattice (`frac`, first digit)** | **+0.0001** | ✅ a second channel |
| logit stack over the library | +0.0004 | ✅ |
| composition / ratio features | +0.0005 | ✅ (superseded by TE) |
| lower learning rate (d5 @ 0.01) | +0.0002 | ✅ |
| 3-seed fold averaging | +0.0002 CV / +0.0001 LB | ✅ small |
| TabM w/ `num_emb_type='pwl'` on TE features | 0.9672 solo, +0.00002 to stack | ⬜ |
| k-NN target encoding, k=30/100/300 | +0.00001 (0.936 standalone) | ⬜ |
| second decimal digit, integer-ness | +0.00001 | ⬜ |
| constraint-geometry features (bounds, slack) | +0.00006 | ⬜ sub-threshold |
| regime-aware stacking (missingness interactions) | +0.00004 | ⬜ |
| pairwise TE (2 parameterisations) | −0.0004 / −0.0001 | ❌ |
| multi-resolution TE | −0.0003 | ❌ |
| TE smoothing sweep | +0.00002 | ⬜ |
| NA-indicator features | −0.00001 | ⬜ MCAR |
| enforcing the accounting identity on imputed values | +0.00002 | ⬜ |
| extra library members once a niche is filled | +0.000001 | ⬜ |
| monotone constraints on the screen columns | −0.0003 | ❌ |
| concatenating the original dataset | −0.0001 | ❌ |
| tree depth 9–13 | to −0.0011 | ❌ |
| **pseudo-labeling confident test rows** | **−0.0034** | ❌ worst of the lot |
| naive mean of the library (12 models, incl. two at 0.944) | −0.0012 | ❌ |
| naive mean of the 5-model library in this notebook | +0.00006 | ⬜ depends on the worst member |

*This notebook computes the noise floor, the imputation comparison, target encoding and
the stacking results live — those are the numbers printed above. The remaining rows come
from the same experiments run outside it (bigger library, tuned settings), so magnitudes
differ slightly; signs and verdicts match. Where the two disagree, trust the cells: they
ran on your machine.*

## What I'd take to the next competition

1. **Measure the noise floor first.** Several of the nulls above look like wins on a single
   5-fold split.
2. **Search representations before models.** Target encoding beat every model choice,
   tuning decision and ensembling trick combined. I spent most of my time on the latter.
3. **On synthetic data, ask what the generator did, not just what the features mean.** The
   decimal lattice is not a fact about phone use; it is a fact about how the numbers were
   written. Two of the three things that worked here came from that question.
3. **Augment, don't replace, when imputing for a NaN-native model.**
4. **If a combiner can only add, it will discard weak-but-diverse models.** Try a linear
   stacker before writing them off.
5. **Predict your LB from your CV before submitting.** It turns "is this leaking?" into a
   falsifiable question — and it's how we validated target encoding.
6. **Pull the source dataset early**, as a diagnostic. It was worth −0.0001 as data and
   corrected our understanding of the whole dataset.
7. **Don't explain a result you haven't tested.** My tidy story about `work_study_hours`
   survived three sections before the source data killed it.

Corrections and disagreements welcome in the comments — I'd rather know.